In [9]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# Parameters

In [34]:
# Load CSV file (replace with the file path in Google Colab)
file_path = '/content/Vesicles_Type_A_Position.csv'  # Update this if necessary
df = pd.read_csv(file_path, skiprows=3)

# Extract relevant columns
id_column = 'ID'
cell_id_column = 'CellID'
filename_column = 'Original Image Name'
x_column = 'Vesicles Type A Position X'
y_column = 'Vesicles Type A Position Y'
z_column = 'Vesicles Type A Position Z'
neighbors = 5

# Filter out rows with missing coordinate values
df = df.dropna(subset=[x_column, y_column, z_column])

# Functions

In [35]:
def calculate_nearest_neighbor(df, filename_column, cell_id_column,
                               x_column, y_column, z_column, id_column, neighbors):
    """
    Calculate nearest neighbor distances for particles within each cell in a dataset.

    Parameters:
        df (pd.DataFrame): Input dataframe with vesicle data.
        filename_column (str): Column name for image/filename.
        cell_id_column (str): Column name for cell ID.
        x_column (str): Column name for X-coordinate.
        y_column (str): Column name for Y-coordinate.
        z_column (str): Column name for Z-coordinate.
        id_column (str): Column name for particle ID.
        neighbors (int): Maximum number of nearest neighbors to consider.

    Returns:
        pd.DataFrame: DataFrame with nearest neighbor distances for each particle.
    """
    results = []

    # Group by 'Original Image Name' and 'CellID'
    grouped = df.groupby([filename_column, cell_id_column])

    for (filename, cell_id), group in grouped:
      coords = group[[x_column, y_column, z_column]].values
      # Build a KD-Tree for the coordinates
      tree = cKDTree(coords)

      # Query distances to the nearest neighbors (including self)
      distances, _ = tree.query(coords, k=neighbors + 1)  # k=neighbor+1 because the first neighbor is the point itself

      for idx, particle_id in enumerate(group[id_column]):
          # Create a result dictionary for each particle
          result = {
              'Original Image Name': filename,
              'CellID': cell_id,
              'ID': particle_id
          }
          # Add nearest neighbor distances as separate columns
          for neighbor in range(1, neighbors + 1):
              result[f'NearestNeighborDistance_{neighbor}'] = distances[idx, neighbor]

          results.append(result)

    return pd.DataFrame(results)

# Data processing

In [36]:
# Calculate nearest neighbor distances
nn_distances_df = calculate_nearest_neighbor(df, filename_column, cell_id_column,
                               x_column, y_column, z_column, id_column, neighbors)

# Calculate median nearest neighbor distances per filename
median_nn_distances = nn_distances_df.groupby('Original Image Name').median().reset_index()

# Save results to CSV
nn_distances_df.to_csv('/content/nearest_neighbor_distances.csv', index=False)
median_nn_distances.to_csv('/content/median_nearest_neighbor_distances.csv', index=False)